In [1]:
import pandas as pd
import duckdb

df_device_alarm_log = pd.DataFrame({
    "alarm_id": [
        501, 502, 503, 504, 505,
        601, 602, 603, 604, 605,
        701, 702, 703, 704,
        801, 802, 803
    ],
    "device_id": [
        "R05", "R05", "R05", "R05", "R05",
        "R16", "R16", "R16", "R16", "R16",
        "R34", "R34", "R34", "R34",
        "R36", "R36", "R36"
    ],
    "alarm_time": [
        "2026-07-26 08:00:00",
        "2026-07-26 09:00:00",
        "2026-07-26 10:00:00",
        "2026-07-26 10:00:00",
        "2026-07-26 11:00:00",

        "2026-07-26 08:00:00",
        "2026-07-26 09:00:00",
        "2026-07-26 09:00:00",
        "2026-07-26 10:00:00",
        "2026-07-26 11:00:00",

        "2026-07-26 07:00:00",
        "2026-07-26 08:00:00",
        "2026-07-26 09:00:00",
        "2026-07-26 10:00:00",

        "2026-07-26 08:00:00",
        "2026-07-26 09:00:00",
        "2026-07-26 10:00:00"
    ],
    "alarm_level": [
        "WARNING", "INFO", "ERROR", "WARNING", "ERROR",
        "ERROR", "WARNING", "ERROR", "INFO", "WARNING",
        "INFO", "WARNING", "ERROR", "WARNING",
        "INFO", "ERROR", "ERROR"
    ],
    "record_status": [
        "VALID", "VALID", "VALID", "VALID", "CANCELLED",
        "VALID", "VALID", "VALID", "VALID", "CANCELLED",
        "VALID", "VALID", "VALID", "VALID",
        "VALID", "VALID", "CANCELLED"
    ]
})

df_device_alarm_log["alarm_time"] = pd.to_datetime(
    df_device_alarm_log["alarm_time"]
)

df_device_alarm_log

,alarm_id,device_id,alarm_time,alarm_level,record_status
0,501,R05,2026-07-26 08:00:00,WARNING,VALID
1,502,R05,2026-07-26 09:00:00,INFO,VALID
2,503,R05,2026-07-26 10:00:00,ERROR,VALID
3,504,R05,2026-07-26 10:00:00,WARNING,VALID
4,505,R05,2026-07-26 11:00:00,ERROR,CANCELLED
5,601,R16,2026-07-26 08:00:00,ERROR,VALID
6,602,R16,2026-07-26 09:00:00,WARNING,VALID
7,603,R16,2026-07-26 09:00:00,ERROR,VALID
8,604,R16,2026-07-26 10:00:00,INFO,VALID
9,605,R16,2026-07-26 11:00:00,WARNING,CANCELLED


# SQL Daily Review：每台设备最近两条有效异常记录

## 题目背景

设备会不断产生告警记录。

部分记录属于普通信息，部分记录已经取消。现在需要从有效的异常记录中，找出每台设备最近的两条。

## 题目要求

先筛选同时满足以下条件的记录：

- `record_status = 'VALID'`
- `alarm_level` 为 `WARNING` 或 `ERROR`

然后对每台设备的记录进行排名，保留最近的两条。

### 排名规则

每台设备内部按照以下顺序排名：

1. `alarm_time` 降序；
2. 当 `alarm_time` 相同时，`alarm_id` 较大的记录优先。

### 输出字段

| 字段 | 含义 |
|---|---|
| `device_id` | 设备编号 |
| `alarm_id` | 告警编号 |
| `alarm_time` | 告警时间 |
| `alarm_level` | 告警等级 |
| `alarm_rank` | 设备内部的时间排名 |

### 边界说明

如果某台设备只有一条符合条件的记录，则只输出这一条，不需要补足两条。

### 最终排序

按照以下顺序排列：

1. `device_id` 升序；
2. `alarm_rank` 升序。

## 解题要求

- 使用 `ROW_NUMBER()`；
- 使用 `PARTITION BY device_id` 分设备排名；
- 使用 CTE 先生成排名；
- 在外层查询中保留 `alarm_rank <= 2`；
- 不使用相关子查询；
- 不使用 `GROUP BY`。

In [ ]:
query = """
WITH rank_table AS (
    SELECT
        device_id,
        alarm_id,
        alarm_time,
        alarm_level,
        ROW_NUMBER() OVER (
            PARTITION BY device_id
            ORDER BY alarm_time DESC, alarm_id DESC
        ) AS alarm_rank
    FROM df_device_alarm_log
    WHERE record_status = 'VALID'
      AND alarm_level IN ('WARNING', 'ERROR')
)

SELECT
    device_id,
    alarm_id,
    alarm_time,
    alarm_level,
    alarm_rank
FROM rank_table
WHERE alarm_rank <= 2
ORDER BY
    device_id,
    alarm_rank;
"""

df = duckdb.execute(query).fetchdf()
df

,device_id,alarm_id,alarm_time,alarm_level,alarm_rank
0,R05,504,2026-07-26 10:00:00,WARNING,1
1,R05,503,2026-07-26 10:00:00,ERROR,2
2,R16,603,2026-07-26 09:00:00,ERROR,1
3,R16,602,2026-07-26 09:00:00,WARNING,2
4,R34,703,2026-07-26 09:00:00,ERROR,2
5,R34,704,2026-07-26 10:00:00,WARNING,1
6,R36,802,2026-07-26 09:00:00,ERROR,1
